In [ ]:
# 获得经纬度
import csv
from selenium import webdriver
import time
from selenium.webdriver.common.by import By
import re

def latlon(input):
    try:    
        # 火狐浏览器
        loginUrl = 'http://api.map.baidu.com/lbsapi/getpoint/index.html'
        driver=webdriver.Firefox()
        driver.get(loginUrl)
        res = []
        for i in range(len(input)):
            res.append([input[i]])
            # 输入不同地级市市政府
            driver.find_element(by=By.ID , value='localvalue').send_keys(input[i]+'政府')
            # 点击一下搜索标志
            driver.find_element(by=By.ID , value='localsearch').click()
            # 下面一行不能删,等加载
            time.sleep(10)
            matinfo=driver.find_element(by=By.TAG_NAME , value='p').text
            print(matinfo)
            string1 = re.split(r"[：]" , matinfo)[-1]
            print(string1)
            # 获得经纬度
            lon = float(string1.split(",")[0])
            lat = float(string1.split(",")[1])
            driver.find_element(by=By.ID , value='localvalue').clear()
            res[i].extend([lon,lat])            
        return(res)        
    
    except:
        print("failed!")

if __name__ == "__main__":
    ls = []
    with open('地级市列表.csv' , encoding = 'gbk') as f:
        csv_reader = csv.reader(f , delimiter = ',')  
        for row in csv_reader: 
            ls.extend([row[0]])
    f.close()
    res = latlon(input=ls)    

    output = open('地级市经纬度.csv', 'w', newline='')
    writer = csv.writer(output)
    for row in res:
        writer.writerow(row)
    output.close()

Problem reading geckodriver versions: error sending request for url (https://raw.githubusercontent.com/SeleniumHQ/selenium/trunk/common/geckodriver/geckodriver-support.json). Using latest geckodriver version


地址：北京市通州区运河东大街57号
电话：(010)12345
坐标：116.731059,39.91088
116.731059,39.91088
地址：石家庄市长安区中山东路216号
电话：(0311)12345
坐标：114.521403,38.048292
114.521403,38.048292
地址：保定市竞秀区东风西路1号
电话：(0312)12345
坐标：115.471191,38.88054
115.471191,38.88054
地址：秦皇岛市海港区西快速路
电话：(0335)12345
坐标：119.525249,39.894365
119.525249,39.894365
地址：唐山市路北区西山道3号
电话：(0315)2823553
坐标：118.18636,39.637353
118.18636,39.637353
地址：河北省邯郸市联通北路1号
电话：(0310)12345
坐标：114.545794,36.631466
114.545794,36.631466
地址：河北省邢台市襄都区新西街166号
电话：(0319)12345
坐标：114.503767,37.065033
114.503767,37.065033
地址：河北省沧州市运河区解放西路39号
电话：(0317)2023539
坐标：116.845245,38.310484
116.845245,38.310484
地址：承德市双桥区府前路1号承德市行政中心内
电话：(0314)12345
坐标：117.969248,40.959556
117.969248,40.959556
地址：廊坊市广阳区广阳道230号
电话：(0316)2331815
坐标：116.690242,39.544018
116.690242,39.544018
地址：衡水市桃城区育才南大街369号
电话：(0318)2061966
坐标：115.6754,37.745691
115.6754,37.745691
地址：张家口市经开区长城西大街10号
电话：(0313)12345
坐标：114.892465,40.774726
114.892465,40.774726
地址：太原市杏花岭区新建路69号
电话：12345
坐标：112.556197,37.876857
112.556197,37.87

In [27]:
# 获得地理距离倒数矩阵
import threading
import math
import queue

# 子进程要执行的任务
def geomatrix(data_int,data_list,queue1):
    print("Threading " +str(data_int)+" start" )
    print("开始计算 " + data_list[data_int][0] + " 的结果")
    # 计算和其他位置的地理距离
    ra = 6378140 # 赤道半径
    rb = 6356755 # 极半径
    flatten = (ra - rb) / ra

    # 输入的点的经纬度
    radloni = math.radians(data_list[data_int][1])
    radlati = math.radians(data_list[data_int][2])
    res_i = [data_list[data_int][0]] # 写一个位置名称在开头
    for j in range(len(data_list)):
        if j != data_int:
            # 计算列表中第j个点与输入的点的距离
            radlonj = math.radians(data_list[j][1])
            radlatj = math.radians(data_list[j][2])

            pi = math.atan(rb / ra * math.tan(radlati))
            pj = math.atan(rb / ra * math.tan(radlatj))
            x = math.acos(math.sin(pi) * math.sin(pj) + math.cos(pi) * math.cos(pj) * math.cos(radloni - radlonj))            
            c1 = (math.sin(x) - x) * (math.sin(pi) + math.sin(pj)) ** 2 / math.cos(x / 2) ** 2
            c2 = (math.sin(x) + x) * (math.sin(pi) - math.sin(pj)) ** 2 / math.sin(x / 2) ** 2
            dr = flatten / 8 * (c1 -c2)
            distance = ra * (x + dr)
            distance = round(distance / 1000 , 4) # 公里
            res_i.extend([round(1 / distance , 4)]) # 公里的倒数
        else:
            res_i.extend([0]) # 0
    print("Threading " +str(data_int)+" end")
    queue1.put(res_i)
    
# 主进程
if __name__ == '__main__':
    threads = []
    geomatrixls = []
    queue1 = queue.Queue()
    for i in range(len(res)):
        thread = threading.Thread(target=geomatrix , args = (i,res,queue1))
        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    while not queue1.empty():
        geomatrixls.append(queue1.get())
    
    #print(geomatrixls)

    output = open('地级市地理距离倒数矩阵.csv', 'w', newline='')
    writer = csv.writer(output)
    for row in geomatrixls:
        writer.writerow(row)
    output.close()

Threading 0 start
开始计算 北京市 的结果
Threading 0 end
Threading 1 start
开始计算 石家庄市 的结果
Threading 1 end
Threading 2 start
开始计算 保定市 的结果
Threading 2 end
Threading 3 start
开始计算 秦皇岛市 的结果
Threading 3 end
Threading 4 start
开始计算 唐山市 的结果
Threading 4 end
Threading 5 start
开始计算 邯郸市 的结果
Threading 5 end
Threading 6 start
开始计算 邢台市 的结果
Threading 6 end
Threading 7 start
开始计算 沧州市 的结果
Threading 7 end
Threading 8 start
开始计算 承德市 的结果
Threading 8 end
Threading 9 start
开始计算 廊坊市 的结果
Threading 9 end
Threading 10 start
开始计算 衡水市 的结果
Threading 10 end
Threading 11 start
开始计算 张家口市 的结果
Threading 11 end
Threading 12 start
开始计算 太原市 的结果
Threading 12 end
Threading 13 start
开始计算 大同市 的结果
Threading 13 end
Threading 14 start
开始计算 阳泉市 的结果
Threading 14 end
Threading 15 start
开始计算 长治市 的结果
Threading 15 end
Threading 16 start
开始计算 临汾市 的结果
Threading 16 end
Threading 17 start
开始计算 晋中市 的结果
Threading 17 end
Threading 18 start
开始计算 运城市 的结果
Threading 18 end
Threading 19 start
开始计算 晋城市 的结果
Threading 19 end
Threading 20 start
开始计算 忻州市 的结果
Threa

In [ ]:
# 获得经纬度
import csv
from selenium import webdriver
import time
from selenium.webdriver.common.by import By
import re

def latlon(input):
    try:    
        # 火狐浏览器
        loginUrl = 'http://api.map.baidu.com/lbsapi/getpoint/index.html'
        driver=webdriver.Firefox()
        driver.get(loginUrl)
        res = []
        for i in range(len(input)):
            res.append([input[i]])
            # 输入不同地级市市政府
            driver.find_element(by=By.ID , value='localvalue').send_keys(input[i]+'政府')
            # 点击一下搜索标志
            driver.find_element(by=By.ID , value='localsearch').click()
            # 下面一行不能删,等加载
            time.sleep(10)
            matinfo=driver.find_element(by=By.TAG_NAME , value='p').text
            print(matinfo)
            string1 = re.split(r"[：]" , matinfo)[-1]
            print(string1)
            # 获得经纬度
            lon = float(string1.split(",")[0])
            lat = float(string1.split(",")[1])
            driver.find_element(by=By.ID , value='localvalue').clear()
            res[i].extend([lon,lat])            
        return(res)        
    
    except:
        print("failed!")

if __name__ == "__main__":
    ls = []
    with open('省级列表.csv' , encoding = 'utf-8-sig') as f:
        csv_reader = csv.reader(f , delimiter = ',')  
        for row in csv_reader: 
            ls.extend([row[0]])
    f.close()
    res = latlon(input=ls)
    output = open('省级经纬度.csv', 'w', newline='',encoding='utf-8-sig')
    writer = csv.writer(output)
    for row in res:
        writer.writerow(row)
    output.close()    

Problem reading geckodriver versions: error sending request for url (https://raw.githubusercontent.com/SeleniumHQ/selenium/trunk/common/geckodriver/geckodriver-support.json). Using latest geckodriver version
There was an error managing geckodriver (error sending request for url (https://github.com/mozilla/geckodriver/releases/latest)); using driver found in the cache


地址：北京市通州区运河东大街57号
电话：(010)12345
坐标：116.731059,39.91088
116.731059,39.91088
地址：天津市河西区友谊路30号
电话：(022)12345
坐标：117.208087,39.091091
117.208087,39.091091
地址：河北省石家庄市长安区裕华东路113号
电话：(0311)12345
坐标：114.537061,38.043517
114.537061,38.043517
地址：山西省太原市小店区省府街3号
电话：(0351)12345
坐标：112.585282,37.819674
112.585282,37.819674
地址：内蒙古自治区呼和浩特市赛罕区敕勒川大街1号
电话：(0471)6944114
坐标：111.771629,40.824303
111.771629,40.824303
地址：沈阳市皇姑区北陵大街45-9号
电话：(024)12345
坐标：123.441771,41.842499
123.441771,41.842499
地址：长春市宽城区新发路329号
电话：(0431)88912321
坐标：125.332284,43.90266
125.332284,43.90266
地址：哈尔滨市南岗区中山路202号
电话：(0451)12345
坐标：126.668637,45.74793
126.668637,45.74793
地址：上海市黄浦区人民大道200号
电话：(021)23111111
坐标：121.480248,31.236276
121.480248,31.236276
地址：南京市鼓楼区北京西路68号
电话：(025)12345
坐标：118.769746,32.067503
118.769746,32.067503
地址：杭州市西湖区省府路8号
电话：(0571)87052576
坐标：120.159428,30.27277
120.159428,30.27277
地址：合肥市滨湖新区中山路1号
电话：(0551)12345
坐标：117.336623,31.74058
117.336623,31.74058
地址：福建省福州市鼓楼区华林路76号
电话：(0591)87837979
坐标：119.302823,26.106865
119.

UnicodeEncodeError: 'gbk' codec can't encode character '\ufeff' in position 0: illegal multibyte sequence

In [1]:
import threading
import math
import queue
import csv


# 子进程要执行的任务
def geomatrix(data_list):
    # 计算和其他位置的地理距离
    ra = 6378140 # 赤道半径
    rb = 6356755 # 极半径
    flatten = (ra - rb) / ra

    radlonj = 120.216329
    radlatj = 30.252589
    res=[]
    for i in range(len(data_list)):
        res.append([data_list[i][0]]) # 写一个位置名称在开头
        radloni = math.radians(float(data_list[i][1]))
        radlati = math.radians(float(data_list[i][2]))
        # 计算列表中第i个点与输入的点的距离           

        pi = math.atan(rb / ra * math.tan(radlati))
        pj = math.atan(rb / ra * math.tan(radlatj))
        x = math.acos(math.sin(pi) * math.sin(pj) + math.cos(pi) * math.cos(pj) * math.cos(radloni - radlonj))            
        c1 = (math.sin(x) - x) * (math.sin(pi) + math.sin(pj)) ** 2 / math.cos(x / 2) ** 2
        c2 = (math.sin(x) + x) * (math.sin(pi) - math.sin(pj)) ** 2 / math.sin(x / 2) ** 2
        dr = flatten / 8 * (c1 -c2)
        distance = ra * (x + dr)
        distance = round(distance / 1000 , 10) # 公里
        #res[i].extend([round(1 / distance , 10)]) # 公里的倒数
        res[i].extend([distance]) 
    return res

    
# 主进程
if __name__ == '__main__':
    ls = []
    with open('省级经纬度.csv' , encoding = 'utf-8-sig') as f:
        csv_reader = csv.reader(f , delimiter = ',')  
        for row in csv_reader: 
            ls.extend([row])
    f.close()
    print(ls)

    geomatrixls = geomatrix(ls)

    output = open('省级距离倒数.csv', 'w', newline='', encoding = 'utf-8-sig')
    writer = csv.writer(output)
    for row in geomatrixls:
        writer.writerow(row)
    output.close()

[['北京市', '116.731059', '39.91088'], ['天津市', '117.208087', '39.091091'], ['河北省', '114.537061', '38.043517'], ['山西省', '112.585282', '37.819674'], ['内蒙古自治区', '111.771629', '40.824303'], ['辽宁省', '123.441771', '41.842499'], ['吉林省', '125.332284', '43.90266'], ['黑龙江省', '126.668637', '45.74793'], ['上海市', '121.480248', '31.236276'], ['江苏省', '118.769746', '32.067503'], ['浙江省', '120.159428', '30.27277'], ['安徽省', '117.336623', '31.74058'], ['福建省', '119.302823', '26.106865'], ['江西省', '115.820878', '28.644512'], ['山东省', '117.027188', '36.676048'], ['河南省', '113.759527', '34.773296'], ['湖北省', '114.348023', '30.552286'], ['湖南省', '112.989635', '28.12162'], ['广东省', '113.272808', '23.139212'], ['广西省', '108.334137', '22.822485'], ['海南省', '110.355294', '20.02449'], ['重庆省', '106.556901', '29.570045'], ['四川省', '104.082823', '30.657042'], ['贵州省', '106.711957', '26.606065'], ['云南省', '102.715798', '25.052745'], ['西藏省', '91.124051', '29.654998'], ['陕西省', '108.960421', '34.272911'], ['甘肃省', '103.833182', '36.06645